# Construcción de los datos

In [16]:
# Construcción de features 
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np

# Cargar csv limpios
orders = pd.read_csv('../../data/processed/olist_orders_clean.csv', parse_dates=[
    'order_purchase_timestamp','order_approved_at',
    'order_delivered_carrier_date','order_delivered_customer_date',
    'order_estimated_delivery_date'
])
items   = pd.read_csv('../../data/processed/olist_order_items_clean.csv', parse_dates=['shipping_limit_date'])
pays    = pd.read_csv('../../data/processed/olist_order_payments_clean.csv')
reviews = pd.read_csv('../../data/processed/olist_order_reviews_clean.csv', parse_dates=[
    'review_creation_date','review_answer_timestamp'
])
prods   = pd.read_csv('../../data/processed/olist_products_clean.csv')
cat_tr  = pd.read_csv('../../data/processed/product_category_name_translation_clean.csv')
sellers = pd.read_csv('../../data/processed/olist_sellers_clean.csv')
customers = pd.read_csv('../../data/processed/olist_customers_clean.csv')


C:\Users\Lizsa\AppData\Local\Temp\ipykernel_14728\1442315200.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders = pd.read_csv('../../data/processed/olist_orders_clean.csv', parse_dates=[
C:\Users\Lizsa\AppData\Local\Temp\ipykernel_14728\1442315200.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders = pd.read_csv('../../data/processed/olist_orders_clean.csv', parse_dates=[
C:\Users\Lizsa\AppData\Local\Temp\ipykernel_14728\1442315200.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders = pd.read_csv('../../data/processed/olist_orders_clean.csv', parse_

In [ ]:
# ORDERS (tiempos, estacionalidad, buckets)

import numpy as np

shape_before_orders = orders.shape

# Asegurar dtype datetime ANTES de operar
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for c in date_cols:
    if c in orders.columns and not pd.api.types.is_datetime64_any_dtype(orders[c]):
        orders[c] = pd.to_datetime(orders[c], errors='coerce')

# Copia para derivar features
fo = orders.copy()

# Tiempos logísticos
fo['delivery_days']       = (fo['order_delivered_customer_date'] - fo['order_purchase_timestamp']) / pd.Timedelta(days=1)
fo['estimated_days']      = (fo['order_estimated_delivery_date'] - fo['order_purchase_timestamp']) / pd.Timedelta(days=1)
fo['delay_vs_estimated']  = (fo['order_delivered_customer_date'] - fo['order_estimated_delivery_date']) / pd.Timedelta(days=1)
fo['on_time']             = fo['order_delivered_customer_date'].notna() & (fo['delay_vs_estimated'] <= 0)
fo['late_days']           = np.where(fo['delay_vs_estimated'].notna(), np.maximum(fo['delay_vs_estimated'], 0), np.nan)
fo['prep_hours']          = (fo['order_delivered_carrier_date'] - fo['order_approved_at']) / pd.Timedelta(hours=1)
fo['transit_days']        = (fo['order_delivered_customer_date'] - fo['order_delivered_carrier_date']) / pd.Timedelta(days=1)

# Calendario / estacionalidad
fo['order_year']    = fo['order_purchase_timestamp'].dt.year
fo['order_month']   = fo['order_purchase_timestamp'].dt.to_period('M').astype(str)  # 'AAAA-MM'
fo['order_week']    = fo['order_purchase_timestamp'].dt.isocalendar().week
fo['order_dow']     = fo['order_purchase_timestamp'].dt.dayofweek                   # 0=Lun … 6=Dom
fo['purchase_hour'] = fo['order_purchase_timestamp'].dt.hour
fo['is_weekend_purchase'] = fo['order_dow'].isin([5, 6])

# Buckets 
fo['delay_bucket'] = pd.cut(
    fo['delay_vs_estimated'],
    bins=[-np.inf, 0, 3, 7, np.inf],
    labels=['a_tiempo_≤0d','1–3d','4–7d','>7d'],
    include_lowest=True
)
fo['delivery_days_bucket'] = pd.cut(
    fo['delivery_days'],
    bins=[-np.inf, 2, 5, 10, np.inf],
    labels=['≤2d','3–5d','6–10d','>10d'],
    include_lowest=True
)

shape_after_orders = fo.shape


C:\Users\Lizsa\AppData\Local\Temp\ipykernel_14728\1467999233.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders[c] = pd.to_datetime(orders[c], errors='coerce')
C:\Users\Lizsa\AppData\Local\Temp\ipykernel_14728\1467999233.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders[c] = pd.to_datetime(orders[c], errors='coerce')
C:\Users\Lizsa\AppData\Local\Temp\ipykernel_14728\1467999233.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders[c] = pd.to_datetime(orders[c], errors='coerce')


In [19]:
# ORDER_ITEMS 
shape_before_items = items.shape
fi_items_agg = items.groupby('order_id').agg(
    item_count=('order_id','size'),
    seller_count=('seller_id','nunique'),
    order_price_total=('price','sum'),
    order_freight_total=('freight_value','sum')
).reset_index()
fi_items_agg['order_total'] = fi_items_agg['order_price_total'] + fi_items_agg['order_freight_total']
fi_items_agg['price_per_item'] = np.where(fi_items_agg['item_count']>0,
                                          fi_items_agg['order_price_total']/fi_items_agg['item_count'],
                                          np.nan)
fi_items_agg['freight_to_price_ratio'] = np.where(fi_items_agg['order_price_total']>0,
                                                  fi_items_agg['order_freight_total']/fi_items_agg['order_price_total'],
                                                  np.nan)
fi_items_agg['mono_seller'] = (fi_items_agg['seller_count'] == 1)
shape_after_items = fi_items_agg.shape

In [ ]:
# ORDER_PAYMENTS 
shape_before_pays = pays.shape

# Método principal por orden 
def _mode_or_nan(s: pd.Series):
    m = s.mode(dropna=True)
    return m.iloc[0] if len(m) else np.nan

fp_pay_agg = pays.groupby('order_id').agg(
    payment_value_total=('payment_value','sum'),
    payment_methods_count=('payment_type','nunique'),
    max_installments=('payment_installments','max'),
    payment_rows=('order_id','size'),
    primary_payment_type=('payment_type', _mode_or_nan)
).reset_index()
fp_pay_agg['multi_payment'] = fp_pay_agg['payment_rows'] > 1

# Bucket de cuotas 
def _bucket_installments(x):
    if pd.isna(x): return np.nan
    x = int(x)
    if x == 1: return '1'
    if 2 <= x <= 6: return '2–6'
    if 7 <= x <= 12: return '7–12'
    if 13 <= x <= 24: return '13–24'
    return '>24'
fp_pay_agg['installment_bucket'] = fp_pay_agg['max_installments'].map(_bucket_installments)

shape_after_pays = fp_pay_agg.shape

In [21]:
# ORDER_REVIEWS 
shape_before_reviews = reviews.shape
fr = reviews[['order_id','review_score','review_creation_date','review_answer_timestamp']].copy()
fr['review_low']   = fr['review_score'] <= 2
fr['review_high']  = fr['review_score'] >= 4
fr['time_to_answer_hours'] = (
    (fr['review_answer_timestamp'] - fr['review_creation_date'])/pd.Timedelta(hours=1)
)
# (Opcional, requiere join con ORDERS → hacerlo en 3.d)
# fr['review_after_delivery_hours'] = ...

shape_after_reviews = fr.shape

In [22]:
# PRODUCTS

shape_before_prods = prods.shape

fp = prods.merge(cat_tr, on='product_category_name', how='left')  # agrega product_category_name_english
# Asegurar numéricos
for c in ['product_length_cm','product_height_cm','product_width_cm','product_weight_g']:
    fp[c] = pd.to_numeric(fp[c], errors='coerce')

# Métricas físicas
fp['volume_cm3']     = fp['product_length_cm'] * fp['product_height_cm'] * fp['product_width_cm']
fp['weight_kg']      = fp['product_weight_g'] / 1000
fp['density_g_cm3']  = np.where(fp['volume_cm3']>0, fp['product_weight_g']/fp['volume_cm3'], np.nan)

shape_after_prods = fp.shape

# Resultados

In [ ]:
def peek(name, df):
    print(f"\n=== {name} ===  shape={df.shape}")
    display(df.head())


In [24]:
print("ORDERS :", shape_before_orders, "->", shape_after_orders)
print("ITEMS  :", shape_before_items,  "->", shape_after_items)
print("PAYS   :", shape_before_pays,   "->", shape_after_pays)
print("REV    :", shape_before_reviews,"->", shape_after_reviews)
print("PRODS  :", shape_before_prods,  "->", shape_after_prods)

ORDERS : (99441, 8) -> (99441, 23)
ITEMS  : (109092, 7) -> (95406, 9)
PAYS   : (103886, 5) -> (99440, 8)
REV    : (98673, 7) -> (98673, 7)
PRODS  : (32951, 9) -> (32951, 13)


In [25]:
peek("features_orders (fo)", fo)
peek("features_items_agg (fi_items_agg)", fi_items_agg)
peek("features_payments_agg (fp_pay_agg)", fp_pay_agg)
peek("features_reviews (fr)", fr)
peek("features_products_enriched (fp)", fp)


=== features_orders (fo) ===  shape=(99441, 23)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_days,...,prep_hours,transit_days,order_year,order_month,order_week,order_dow,purchase_hour,is_weekend_purchase,delay_bucket,delivery_days_bucket
0,10a045cdf6a5650c21e9cfeb60384c16,a4b417188addbc05b26b72d5e44837a1,canceled,2018-10-17 17:30:18,NaT,NaT,NaT,2018-10-30,NaN,12.270625,...,NaN,NaN,2018,2018-10,42,2,17,False,NaN,NaN
1,b059ee4de278302d550a3035c4cdb740,856336203359aa6a61bf3826f7d84c49,canceled,2018-10-16 20:16:02,NaT,NaT,NaT,2018-11-12,NaN,26.155532,...,NaN,NaN,2018,2018-10,42,1,20,False,NaN,NaN
2,a2ac6dad85cf8af5b0afb510a240fe8c,4c2ec60c29d10c34bd49cb88aa85cfc4,canceled,2018-10-03 18:55:29,NaT,NaT,NaT,2018-10-16,NaN,12.211470,...,NaN,NaN,2018,2018-10,40,2,18,False,NaN,NaN
3,616fa7d4871b87832197b2a137a115d2,bf6181a85bbb4115736c0a8db1a53be3,canceled,2018-10-01 15:30:09,NaT,NaT,NaT,2018-10-23,NaN,21.354063,...,NaN,NaN,2018,2018-10,40,0,15,False,NaN,NaN
4,392ed9afd714e3c74767d0c4d3e3f477,2823ffda607a2316375088e0d00005ec,canceled,2018-09-29 09:13:03,NaT,NaT,NaT,2018-10-15,NaN,15.615937,...,NaN,NaN,2018,2018-09,39,5,9,True,NaN,NaN



=== features_items_agg (fi_items_agg) ===  shape=(95406, 9)


,order_id,item_count,seller_count,order_price_total,order_freight_total,order_total,price_per_item,freight_to_price_ratio,mono_seller
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,58.90,13.29,72.19,58.90,0.225637,True
1,00018f77f2f0320c557190d7a144bdd3,1,1,239.90,19.93,259.83,239.90,0.083076,True
2,000229ec398224ef6ca0657da4fc703e,1,1,199.00,17.87,216.87,199.00,0.089799,True
3,00024acbcdf0a6daa1e931b038114c75,1,1,12.99,12.79,25.78,12.99,0.984604,True
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,199.90,18.14,218.04,199.90,0.090745,True



=== features_payments_agg (fp_pay_agg) ===  shape=(99440, 8)


,order_id,payment_value_total,payment_methods_count,max_installments,payment_rows,primary_payment_type,multi_payment,installment_bucket
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,1,credit_card,False,2–6
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3,1,credit_card,False,2–6
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5,1,credit_card,False,2–6
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2,1,credit_card,False,2–6
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3,1,credit_card,False,2–6



=== features_reviews (fr) ===  shape=(98673, 7)


,order_id,review_score,review_creation_date,review_answer_timestamp,review_low,review_high,time_to_answer_hours
0,bd2c2c3a4d59e68fb14a526745572883,4,2018-08-31,2018-09-01 12:27:54,False,True,36.465000
1,529a65336debb1c3e3327d7d67dc733d,5,2018-08-31,2018-09-01 20:50:38,False,True,44.843889
2,b0e9288a209f5ec50391c140dba4c91f,5,2018-08-31,2018-09-05 22:33:44,False,True,142.562222
3,d02b32c3bcfb76481817b2222b990e84,5,2018-08-31,2018-08-31 23:24:47,False,True,23.413056
4,8506571faf231af2bd4e43d1ba47dce8,5,2018-08-31,2018-08-31 19:07:10,False,True,19.119444



=== features_products_enriched (fp) ===  shape=(32951, 13)


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,volume_cm3,weight_kg,density_g_cm3
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery,2240.0,0.225,0.100446
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art,10800.0,1.000,0.092593
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure,2430.0,0.154,0.063374
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby,2704.0,0.371,0.137204
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares,4420.0,0.625,0.141403


In [26]:
# Check de consistencia: pagos vs totales del pedido
_check = (fi_items_agg[['order_id','order_total']]
          .merge(fp_pay_agg[['order_id','payment_value_total']], on='order_id', how='left'))
_check['diff_pay_total'] = (_check['payment_value_total'] - _check['order_total']).round(2)
display(_check.head())

,order_id,order_total,payment_value_total,diff_pay_total
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,72.19,0.0
1,00018f77f2f0320c557190d7a144bdd3,259.83,259.83,0.0
2,000229ec398224ef6ca0657da4fc703e,216.87,216.87,0.0
3,00024acbcdf0a6daa1e931b038114c75,25.78,25.78,0.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,218.04,-0.0


# Guardado

In [28]:
# Guardado
outdir = '../../data/features/'
os.makedirs(outdir, exist_ok=True)

fo.to_csv(outdir + 'features_orders.csv', index=False)
fi_items_agg.to_csv(outdir + 'features_items_agg.csv', index=False)
fp_pay_agg.to_csv(outdir + 'features_payments_agg.csv', index=False)
fr.to_csv(outdir + 'features_reviews.csv', index=False)
fp.to_csv(outdir + 'features_products_enriched.csv', index=False)

print("Features guardados en:", outdir)


Features guardados en: ../../data/features/
